In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT = "./V5C_Final_Merged_Model"


# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5B MODEL")
print("=" * 100)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    trust_remote_code=True,
)

print("Loading merged model...")

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("✅ V5B Model Loaded Successfully")


# ==========================================================
# SYSTEM PROMPT FOR V5B
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V5B.

Your ONLY task is to predict the best communication strategy
for the assistant in the current conversation.

V5B predicts HOW the assistant should communicate,
not WHAT the assistant should say.

You must output EXACTLY one JSON object containing
EXACTLY these four keys:

{
  "tone": "",
  "communication_style": "",
  "detail_level": "",
  "approach": ""
}

FIELD DEFINITIONS:

tone:
The appropriate emotional/professional tone for the response.

communication_style:
The overall style the assistant should use.

detail_level:
How much detail the response should contain.

approach:
The overall communication approach the assistant should take.

RULES:

1. Predict HOW the assistant should communicate.
2. Do NOT generate the final assistant reply.
3. Do NOT generate suggested messages.
4. Do NOT predict assistant objectives.
5. Do NOT extract memories.
6. Do NOT summarize the conversation.
7. Do NOT analyze compatibility.
8. Do NOT analyze personality.
9. Do NOT provide the actual advice or solution.
10. Do NOT add any fields other than the four required fields.
11. Use the conversation as the primary evidence.
12. Use the relationship context to select an appropriate communication style.
13. Do not infer unsupported communication styles.
14. Keep the strategy appropriate to the user's current situation.
15. Return JSON only.
16. Do not output markdown.
17. Do not output <think>.
18. Your output must begin with { and end with }.

The four required fields are:

"tone"
"communication_style"
"detail_level"
"approach"
"""


# ==========================================================
# TEST CONVERSATIONS
# ==========================================================

test_conversations = [

    {
        "name": "V5B TEST 1 - Emotional Support",

        "domain": "relationship",

        "relationship": "Partner",

        "conversation": """
Partner: You seem overwhelmed lately.

User: I've been stressed with work and don't know where to start.

Partner: You've had a lot going on recently.

User: Yeah. I feel like everything is piling up at once.
"""
    },


    {
        "name": "V5B TEST 2 - Problem Solving",

        "domain": "relationship",

        "relationship": "Manager",

        "conversation": """
Manager: Have you completed the report?

User: Not yet. I'm having trouble analyzing the data.

Manager: What part are you stuck on?

User: I can't figure out why the numbers don't match the previous report.
"""
    },


    {
        "name": "V5B TEST 3 - Decision Support",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: Which laptop should I buy?

User: I'm looking for one for programming and college.

Friend: Do you have a budget?

User: Around $1,200. I want something reliable that will last several years.
"""
    },


    {
        "name": "V5B TEST 4 - Anxiety / Reassurance",

        "domain": "relationship",

        "relationship": "Close Friend",

        "conversation": """
Friend: How are you feeling about your interview tomorrow?

User: Honestly, I'm really nervous.

Friend: You've prepared a lot though.

User: I know, but I keep thinking I'm going to mess up and forget everything.
"""
    },


    {
        "name": "V5B TEST 5 - Excited Sharing",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: Did you hear back from the company?

User: Yes! I got the job!

Friend: That's amazing!

User: I can't believe it. I've been waiting for this opportunity for months.
"""
    },


    {
        "name": "V5B TEST 6 - Disappointment",

        "domain": "relationship",

        "relationship": "Partner",

        "conversation": """
Partner: Did you hear back from the hiring manager?

User: Yeah. They rejected me.

Partner: I'm really sorry.

User: I honestly thought the final interview went really well.
I'm pretty disappointed right now.
"""
    },


    {
        "name": "V5B TEST 7 - Neutral Information",

        "domain": "relationship",

        "relationship": "Colleague",

        "conversation": """
Colleague: Did you finish the spreadsheet?

User: Yes. I updated the numbers in column D and uploaded
the latest version to the shared folder.

Colleague: Great, thanks.
"""
    },


    {
        "name": "V5B TEST 8 - Complex Problem",

        "domain": "relationship",

        "relationship": "Colleague",

        "conversation": """
Colleague: The deployment failed again.

User: I checked the logs and found several errors.

Colleague: Do you know what caused it?

User: The API container is failing during startup because
the environment variable isn't being loaded correctly.

Colleague: Can you figure out how we should fix it?

User: I think we need to check the deployment configuration
and compare it with the previous working version.
"""
    },

]


# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    # ------------------------------------------------------
    # USER PROMPT
    # ------------------------------------------------------

    user_prompt = f"""
Domain: {test["domain"]}

Relationship: {test["relationship"]}

Conversation:

{test["conversation"]}

Instruction:
Predict assistant communication strategy.
"""

    # ------------------------------------------------------
    # CHAT MESSAGES
    # ------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
        )
    
    # ------------------------------------------------------
    # DECODE ONLY NEW TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    # ------------------------------------------------------
    # PRINT RESULT
    # ------------------------------------------------------

    print("\nMODEL OUTPUT")
    print("-" * 100)
    print(response)

    print("\n")

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


LOADING V5B MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 5/5 [01:11<00:00, 14.25s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ V5B Model Loaded Successfully


V5B TEST 1 - Emotional Support


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...


MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"tone":"Excited","communication_style":"Supportive","detail_level":"Brief","approach":"Validate the user's excitement and offer congratulations."}




V5B TEST 6 - Disappointment


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

{"tone":"Empathetic","communication_style":"Supportive","detail_level":"Moderate","approach":"Validate the user's feelings and offer emotional reassurance."}




V5B TEST 7 - Neutral Information


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"tone":"Professional","communication_style":"Informative","detail_level":"Brief","approach":"Confirm the task completion status concisely."}




V5B TEST 8 - Complex Problem

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

{"tone":"Professional","communication_style":"Solution-Oriented","detail_level":"Moderate","approach":"Focus on practical solutions and technical troubleshooting steps."}




Testing the V4A capacity in v5b modle

In [ ]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT = "./V5B_Final_Merged_Model"


# ==========================================================
# LOAD 4-BIT CONFIG
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# ==========================================================
# LOAD MODEL
# ==========================================================

print("=" * 100)
print("LOADING V5B MODEL")
print("=" * 100)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    trust_remote_code=True,
)

print("Loading V5B model...")

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("✅ V5B Model Loaded Successfully")


# ==========================================================
# V4A TASK PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V4A.

Your ONLY task is to extract stable long-term memories about the USER.

Keep ONLY information that is likely to remain useful across future
conversations.

KEEP:

- Long-term preferences
- Long-term goals
- Stable personality traits
- Persistent habits
- Profession
- Skills
- Communication preferences
- Ongoing projects
- Persistent health conditions

IGNORE:

- Temporary emotions
- Greetings
- One-time events
- Short-term plans
- Temporary problems
- Casual conversation
- Temporary feelings
- Temporary situations

IMPORTANT:

A memory must describe the USER.

Do not extract memories about other people.

Do not generate a reply.

Do not give advice.

Do not analyze the relationship.

Do not predict the future.

Do not generate communication strategy.

Return ONLY valid JSON.

Output format:

{
  "memories": [
    "...",
    "...",
    "..."
  ]
}

If there are no stable long-term memories:

{
  "memories": []
}

Rules:

1. Extract only stable long-term user information.
2. Do not include temporary emotions.
3. Do not include one-time events.
4. Do not include short-term plans.
5. Do not include relationship information.
6. Do not include memories about other people.
7. Do not summarize the conversation.
8. Do not generate an assistant reply.
9. Do not add extra JSON fields.
10. Return JSON only.
11. Do not output markdown.
12. Do not output <think>.
13. Output exactly one JSON object.
"""


# ==========================================================
# TEST DATA
# ==========================================================

test_conversations = [

    # ======================================================
    # TEST 1
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 1 - Career Goals",

        "domain": "career",
        "relationship": "Career Coach",

        "conversation": """
Career Coach: What are your long-term career goals?

User: I want to become a Principal Data Scientist.

Career Coach: Do you want to become a manager eventually?

User: No. I want to remain an individual contributor.

Career Coach: What type of work do you enjoy most?

User: I really enjoy machine learning, technical architecture,
and explaining complex technical findings to stakeholders.

Career Coach: Is that something you want to continue long term?

User: Definitely. I want to keep growing technically rather than
moving into people management.
"""
    },


    # ======================================================
    # TEST 2
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 2 - Communication Preference",

        "domain": "work",
        "relationship": "Manager",

        "conversation": """
Manager: How do you prefer to receive routine updates?

User: I strongly prefer asynchronous written communication.

Manager: Why?

User: I like having everything documented. I don't like having
meetings just for simple status updates.

Manager: Would you prefer written updates even for small tasks?

User: Yes. For routine matters, written communication is much
better for me.
"""
    },


    # ======================================================
    # TEST 3
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 3 - Persistent Work Habit",

        "domain": "work",
        "relationship": "Colleague",

        "conversation": """
Colleague: When do you usually do your most difficult work?

User: I normally block my mornings for deep technical work.

Colleague: Every morning?

User: Almost every workday.

Colleague: Why mornings?

User: I need uninterrupted quiet time. Meetings and Slack messages
during that period destroy my concentration.

Colleague: So protecting your mornings is important?

User: Very important. I usually schedule deep analysis before noon.
"""
    },


    # ======================================================
    # TEST 4
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 4 - Long-Term + Temporary Information",

        "domain": "career",
        "relationship": "Career Coach",

        "conversation": """
Career Coach: How has your week been?

User: Honestly, this week has been exhausting.

Career Coach: Why?

User: I barely slept last night because I was worried about
my presentation this morning.

Career Coach: How did the presentation go?

User: Surprisingly well. I was nervous before it started, but
I felt confident once I got into the technical details.

Career Coach: What did you enjoy?

User: Explaining the data and presenting technical findings
to stakeholders.

Career Coach: Is that something you want to continue?

User: Yes. I really enjoy presenting complex analysis to stakeholders.

Career Coach: What are your long-term goals?

User: I want to become a Principal Data Scientist while remaining
an individual contributor.

User: I don't want to move into people management.

Career Coach: What about your work schedule?

User: I usually block my mornings for deep technical work.

Career Coach: Anything else?

User: I prefer asynchronous written communication for routine updates.

User: Tomorrow I'm taking the afternoon off to visit my family.

User: Also, my laptop battery died yesterday, but it's fixed now.
"""
    },


    # ======================================================
    # TEST 5
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 5 - Personality + Preferences",

        "domain": "personal",
        "relationship": "Friend",

        "conversation": """
Friend: How would you describe yourself?

User: I'm pretty introverted.

Friend: Do you enjoy large social events?

User: Not really. I prefer small groups and one-on-one conversations.

Friend: What do you usually do on weekends?

User: I usually read, work on personal programming projects,
or spend quiet time at home.

Friend: Do you like spontaneous plans?

User: Not particularly. I prefer having plans ahead of time.
"""
    },


    # ======================================================
    # TEST 6
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 6 - Skills + Ongoing Project",

        "domain": "technology",
        "relationship": "Teammate",

        "conversation": """
Teammate: What are you currently working on?

User: I'm building a predictive analytics system for our team.

Teammate: Is that just a short experiment?

User: No. It's an ongoing project that I've been working on
for several months and expect to continue throughout the year.

Teammate: What technologies do you normally work with?

User: Mostly Python, PyTorch, and machine learning systems.

Teammate: So those are some of your strongest technical skills?

User: Yes. Machine learning engineering is one of my strongest areas.
"""
    },


    # ======================================================
    # TEST 7
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 7 - Values / Career Preferences",

        "domain": "career",
        "relationship": "Career Coach",

        "conversation": """
Career Coach: Are there industries you would prefer to avoid?

User: Yes. I don't want to work for defense contractors
or tobacco companies.

Career Coach: Is that temporary?

User: No. Those industries don't align with my personal values.

Career Coach: What kind of companies interest you instead?

User: Companies where I can have meaningful technical impact
and work on products that align with my values.
"""
    },


    # ======================================================
    # TEST 8
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 8 - Temporary Information Only",

        "domain": "personal",
        "relationship": "Friend",

        "conversation": """
Friend: What are you doing tomorrow?

User: I'm taking the afternoon off to visit my family.

Friend: Are you taking the whole day?

User: No, just tomorrow afternoon.

Friend: How are you feeling today?

User: I'm exhausted. I barely slept last night.

Friend: Did something happen?

User: I had a stressful presentation this morning.

Friend: How did it go?

User: It went fine. I'm just tired now.
"""
    },


    # ======================================================
    # TEST 9
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 9 - Mixed Strong Memory Signals",

        "domain": "career",
        "relationship": "Mentor",

        "conversation": """
Mentor: Tell me about the direction you want your career to take.

User: I want to become a Principal Data Scientist.

Mentor: Management?

User: No. I want to remain an individual contributor.

Mentor: What are your strongest skills?

User: Machine learning, Python, PyTorch, technical architecture,
and explaining complex analysis.

Mentor: How do you prefer to communicate at work?

User: I prefer asynchronous written communication for routine matters.

Mentor: How do you structure your workday?

User: I protect my mornings for uninterrupted deep technical work.

Mentor: What are you working on currently?

User: A predictive analytics system that I've been developing
for several months.

Mentor: Anything about your career values?

User: I don't want to work in industries that conflict with
my personal values.
"""
    },


    # ======================================================
    # TEST 10
    # ======================================================

    {
        "name": "V4A CAPABILITY TEST 10 - No Stable Memory",

        "domain": "personal",
        "relationship": "Friend",

        "conversation": """
Friend: How are you doing today?

User: I'm tired.

Friend: Rough day?

User: Yeah, I barely slept last night.

Friend: Are you doing anything tonight?

User: Probably just going to rest.

Friend: Hopefully tomorrow is better.

User: Yeah, hopefully.
"""
    }
]


# ==========================================================
# RUN TESTS
# ==========================================================

for i, test in enumerate(test_conversations, start=1):

    print("\n")
    print("=" * 100)
    print(f"TEST {i}: {test['name']}")
    print("=" * 100)

    user_prompt = f"""
Domain: {test['domain']}

Relationship: {test['relationship']}

Conversation:

{test['conversation']}

Instruction:
Extract stable long-term memories about the USER.
"""

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    # ------------------------------------------------------
    # DECODE ONLY NEW TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # ------------------------------------------------------
    # PRINT RAW OUTPUT
    # ------------------------------------------------------

    print("\nMODEL OUTPUT")
    print("-" * 100)
    print(response)

    # ------------------------------------------------------
    # BASIC V4A FORMAT CHECK
    # ------------------------------------------------------

    print("\nV4A FORMAT CHECK")
    print("-" * 100)

    try:

        parsed = json.loads(response)

        if "memories" in parsed:

            print("✅ 'memories' field found")

            if isinstance(parsed["memories"], list):
                print("✅ memories is a list")

                print(f"Memory count: {len(parsed['memories'])}")

                for memory in parsed["memories"]:
                    print(f"  - {memory}")

            else:
                print("❌ memories is NOT a list")

        else:

            print("❌ Missing 'memories' field")

        extra_keys = set(parsed.keys()) - {"memories"}

        if extra_keys:

            print(f"⚠️ Extra fields detected: {extra_keys}")

        else:

            print("✅ No extra fields")

    except json.JSONDecodeError:

        print("❌ Output is NOT valid JSON")

    print("\n")

/home/ubuntu/V5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING V5B MODEL
Loading tokenizer...
Loading V5B model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.64s/it]
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


✅ V5B Model Loaded Successfully


TEST 1: V4A CAPABILITY TEST 1 - Career Goals

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"memories": ["Wants to become a Principal Data Scientist", "Wants to remain an individual contributor rather than move into management", "Enjoys machine learning, technical architecture, and explaining technical findings to stakeholders"]}

V4A FORMAT CHECK
----------------------------------------------------------------------------------------------------
❌ Output is NOT valid JSON




TEST 2: V4A CAPABILITY TEST 2 - Communication Preference

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"memories": ["Prefers asynchronous written communication for routine updates", "Dislikes meetings for simple status updates", "Prefers written updates for small tasks"]}]}

V4A FORMAT CHECK
--------------